In [ ]:
# !/usr/bin/env python3
# Dieses Skript durchsucht GitHub nach Dateien, die Links zu Trendshift‑Repositories enthalten, und speichert die gefundenen Repositories in einer CSV‑Datei.
# Benötigt wird ein GitHub‑Token, um die API‑Anfragen zu authentifizieren. Das Skript paginiert durch die Suchergebnisse, um alle relevanten Repositories zu erfassen, und speichert die Informationen in einer CSV‑Datei mit den Spalten "id", "name" und "url".

# ctrl+shift+p -> "Python: Run Python File in Terminal" ausführen, um das Skript zu starten.
import requests
import csv
import time

SEARCH_QUERY = '"https://trendshift.io/repositories/"'
GRAPHQL_URL = "https://api.github.com/graphql"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

def run_query(after_cursor=None):
    query = """
    query($queryString: String!, $after: String) {
      search(query: $queryString, type: CODE, first: 50, after: $after) {
        pageInfo {
          hasNextPage
          endCursor
        }
        edges {
          node {
            ... on CodeResult {
              repository {
                id
                nameWithOwner
                url
              }
            }
          }
        }
      }
    }
    """

    variables = {
        "queryString": SEARCH_QUERY,
        "after": after_cursor
    }

    r = requests.post(
        GRAPHQL_URL,
        json={"query": query, "variables": variables},
        headers=HEADERS
    )

    # GitHub erlaubt GraphQL ohne Token, aber limitiert → Retry
    if r.status_code == 403:
        print("⏳ Rate‑Limit – warte 10 Sekunden…")
        time.sleep(10)
        return run_query(after_cursor)

    r.raise_for_status()
    return r.json()

def main():
    repos = {}
    cursor = None

    print("🔍 Starte GitHub‑GraphQL‑Scraping…")

    while True:
        data = run_query(cursor)

        search = data["data"]["search"]
        edges = search["edges"]

        for edge in edges:
            repo = edge["node"]["repository"]
            repos[repo["id"]] = {
                "id": repo["id"],
                "name": repo["nameWithOwner"],
                "url": repo["url"]
            }

        if not search["pageInfo"]["hasNextPage"]:
            break

        cursor = search["pageInfo"]["endCursor"]
        print(f"➡️  Weiter mit Cursor: {cursor}")
        time.sleep(1)

    print(f"✅ Gesamt gefundene Repositories: {len(repos)}")

    with open("trendshift_repos_graphql.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "name", "url"])
        for repo in sorted(repos.values(), key=lambda x: x["id"]):
            writer.writerow([repo["id"], repo["name"], repo["url"]])

    print("📁 CSV gespeichert als trendshift_repos_graphql.csv")

if __name__ == "__main__":
    main()
